# Hugging Face + Transformers для Computer Vision

В этом ноутбуке мы научимся использовать самую популярную библиотеку в мире — **Hugging Face Transformers** — для задач компьютерного зрения.

**Что ты изучишь:**
1. Загрузка предобученных моделей (ViT, Swin, ConvNeXt, BEiT)
2. Использование `pipeline` для быстрого старта
3. Feature extraction (получение эмбеддингов)
4. Fine-tuning на своём датасете
5. Сравнение разных моделей
6. Лучшие практики работы с Hugging Face

> **Запускай в Colab** — Runtime → T4 GPU.

## Почему Hugging Face?

- Более **50 000** предобученных моделей для vision
- Единый API для всех моделей
- Огромное сообщество и документация
- Легко интегрируется с PyTorch
- Поддержка самых современных архитектур (ViT, Swin, ConvNeXt, DINOv2 и др.)

# 1. Установка и импорт

In [ ]:
!pip install --quiet -U transformers timm torchinfo

In [ ]:
import torch
import torch.nn as nn
from transformers import (
    AutoImageProcessor, 
    AutoModelForImageClassification,
    pipeline
)
from PIL import Image
import requests
from io import BytesIO

# 2. Быстрый старт: `pipeline`

Самый простой способ использовать модель — через `pipeline`.

In [ ]:
# Загружаем готовый pipeline для классификации изображений
pipe = pipeline(
    "image-classification", 
    model="google/vit-base-patch16-224",
    device=0 if torch.cuda.is_available() else -1
)

# Пример использования
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(BytesIO(requests.get(url).content))

results = pipe(image)
print(results)

# 3. Загрузка модели и процессора вручную

Для более гибкой работы используем `AutoImageProcessor` и `AutoModelForImageClassification`.

In [ ]:
model_name = "google/vit-base-patch16-224"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

print(f"Модель загружена: {model_name}")
print(f"Количество параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# 4. Feature Extraction (получение эмбеддингов)

Часто нам нужны не предсказания, а **векторные представления** изображений.

In [ ]:
from transformers import AutoModel

# Загружаем модель без головы классификации
feature_extractor = AutoModel.from_pretrained("google/vit-base-patch16-224")

def get_image_embedding(image_path_or_url):
    if image_path_or_url.startswith('http'):
        image = Image.open(BytesIO(requests.get(image_path_or_url).content))
    else:
        image = Image.open(image_path_or_url)
    
    inputs = processor(image, return_tensors="pt")
    
    with torch.no_grad():
        outputs = feature_extractor(**inputs)
    
    # Берём [CLS] токен (первый токен)
    embedding = outputs.last_hidden_state[:, 0, :].squeeze()
    return embedding

emb = get_image_embedding(url)
print(f"Размер эмбеддинга: {emb.shape}")  # [768]

# 5. Fine-tuning на своём датасете

Давайте дообучим ViT на датасете **Bees vs Ants** (из WS3).

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Датасет
train_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

trainset = datasets.ImageFolder('./data/hymenoptera_data/train', train_tfm)
trainloader = DataLoader(trainset, batch_size=16, shuffle=True)

# Модель
model = AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=2,
    ignore_mismatched_sizes=True
)

# Замораживаем backbone (только голова будет обучаться)
for param in model.vit.parameters():
    param.requires_grad = False

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./vit-finetuned",
    per_device_train_batch_size=16,
    num_train_epochs=5,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=trainset,
    eval_dataset=trainset,  # для примера
)

trainer.train()

# 6. Сравнение популярных моделей

In [ ]:
models = [
    "google/vit-base-patch16-224",
    "microsoft/swin-tiny-patch4-window7-224",
    "facebook/convnext-tiny-224",
    "microsoft/beit-base-patch16-224"
]

for m in models:
    try:
        model = AutoModelForImageClassification.from_pretrained(m)
        params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"{m}: {params:.1f}M параметров")
    except:
        print(f"{m}: не удалось загрузить")

# 7. Упражнения

### Упражнение 1: Zero-shot классификация

Используй модель `openai/clip-vit-base-patch32` для zero-shot классификации (без дообучения).

In [ ]:
# Твой код здесь

### Упражнение 2: DINOv2

Загрузи `facebook/dinov2-base` и посмотри, какие эмбеддинги она выдаёт. Сравни с ViT.

In [ ]:
# Твой код здесь

### Упражнение 3 (сложное): Полный fine-tuning

Возьми датасет из WS3 и полностью дообучи ViT или Swin (без заморозки). Сравни результат с замороженной версией.

---

**Готово!**  
Ты научился работать с самой мощной экосистемой предобученных моделей в мире.

Следующий шаг — **WS6: UNet и сегментация**.